# 03 — Comparing Buffer Sizes

A single buffer answers *is this within range?*

Multiple buffers at different radii answer *how close?* — turning a binary yes/no into a gradient of proximity. This is how blast zones, noise contours, and risk maps are structured: concentric rings, each representing a different intensity level.

```
inner ring  →  high intensity   (lethal / severe)
middle ring →  medium intensity (damaging / moderate)
outer ring  →  low intensity    (warning / minor)
```

## Setup

In [1]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "targets.geojson") as f:
    targets_fc = json.load(f)


def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0
    d = distance_km / R
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def point_buffer(lon, lat, radius_km, n_points=64):
    ring = [destination_point(lon, lat, 360.0*i/n_points, radius_km) for i in range(n_points)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

def buffer_feature(lon, lat, radius_km, name="", n_points=64):
    return {
        "type": "Feature",
        "properties": {"name": name, "radius_km": radius_km},
        "geometry": point_buffer(lon, lat, radius_km, n_points)
    }

# Helper: extract lon/lat from a Point feature
def target_coords(name):
    feat = next(f for f in targets_fc["features"] if f["properties"]["name"] == name)
    return feat["geometry"]["coordinates"]

print(f"Loaded {len(targets_fc['features'])} targets.")

Loaded 4 targets.


## Concentric Rings Around One Target

Three radii around the same center produces the nested ring structure. The outer ring contains the inner rings — the area covered by the inner ring is always a subset of the outer ring.

In [2]:
tehran = target_coords("Tehran")

# Three impact zones — radii in km
zones = [
    {"radius_km": 50,  "label": "Lethal",    "color": "#e74c3c"},
    {"radius_km": 150, "label": "Damaging",  "color": "#e67e22"},
    {"radius_km": 300, "label": "Warning",   "color": "#f1c40f"},
]

m = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)

# Add outermost first so inner rings render on top
for zone in reversed(zones):
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{zone['label']} ({zone['radius_km']} km)",
        style={"color": zone["color"], "fillColor": zone["color"],
               "fillOpacity": 0.35, "weight": 1.5}
    ))

m.add(LayersControl())

print("Zone radii:", [z['radius_km'] for z in zones], "km")
m

Zone radii: [50, 150, 300] km


Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Interpreting Zone Overlap

The inner zone is entirely contained within the outer zones. The **area of the outer ring that is not in the inner ring** represents a region of *less* intensity — close enough to be affected, but not in the core zone.

For a given target, we can compute the exclusive area of each band (annular ring area) using the formula for a circle:

```
band area = π × r_outer² − π × r_inner²
```

In [3]:
import math

radii = [0] + [z["radius_km"] for z in zones]   # add 0 as inner bound of first band

print(f"{'Band':<25} {'Exclusive area':>18}")
print("─" * 46)
for i in range(1, len(radii)):
    r_inner = radii[i - 1]
    r_outer = radii[i]
    area_km2 = math.pi * (r_outer**2 - r_inner**2)
    label = f"{zones[i-1]['label']} zone (0–{r_outer} km)"
    print(f"  {label:<23}  {area_km2:>14,.0f} km²")

Band                          Exclusive area
──────────────────────────────────────────────
  Lethal zone (0–50 km)             7,854 km²
  Damaging zone (0–150 km)          62,832 km²
  Warning zone (0–300 km)         212,058 km²


## Overlapping Buffers from Multiple Targets

When two targets are close enough, their buffers overlap. The overlap region is reachable from both — useful for identifying shared risk zones or mutual coverage.

In [4]:
# Compare Tehran and Riyadh — roughly 1,400 km apart
riyadh = target_coords("Riyadh")

multi_target_fc = {
    "type": "FeatureCollection",
    "features": [
        buffer_feature(*tehran, radius_km=500, name="Tehran 500 km"),
        buffer_feature(*riyadh, radius_km=500, name="Riyadh 500 km"),
    ]
}

# Point features for the targets
point_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "properties": {"name": "Tehran"},
         "geometry": {"type": "Point", "coordinates": tehran}},
        {"type": "Feature", "properties": {"name": "Riyadh"},
         "geometry": {"type": "Point", "coordinates": riyadh}},
    ]
}

m2 = Map(center=(35, 47), zoom=4, basemap=basemaps.CartoDB.Positron)
m2.add(GeoJSON(
    data=multi_target_fc,
    style={"color": "#8e44ad", "fillColor": "#8e44ad", "fillOpacity": 0.25, "weight": 1.5}
))
m2.add(GeoJSON(
    data=point_fc,
    style={"color": "#2c3e50", "fillColor": "#2c3e50", "fillOpacity": 0.8, "weight": 1}
))
m2

Map(center=[35, 47], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## Exercise A

Create **three concentric buffers** (75 km, 200 km, 450 km) around **Honolulu** (`[-157.855, 21.305]`).

1. Display all three on a single map with distinct colors.
2. Print the exclusive band area (km²) for each ring.
3. Which band has the largest area, and why?

In [5]:
from ipyleaflet import Map, GeoJSON, basemaps
import math

honolulu = [-157.855, 21.305]

zones = [
    {"radius_km": 75,  "label": "75 km zone",  "color": "#e74c3c"},
    {"radius_km": 200, "label": "200 km zone", "color": "#e67e22"},
    {"radius_km": 450, "label": "450 km zone", "color": "#f1c40f"},
]

# 1. Display all three on one map
m = Map(
    center=(honolulu[1], honolulu[0]),
    zoom=4,
    basemap=basemaps.CartoDB.Positron
)

# Add largest first so smaller buffers are visible on top
for zone in reversed(zones):
    feat = buffer_feature(
        *honolulu,
        radius_km=zone["radius_km"],
        name=zone["label"]
    )

    m.add(GeoJSON(
        data={
            "type": "FeatureCollection",
            "features": [feat]
        },
        name=zone["label"],
        style={
            "color": zone["color"],
            "fillColor": zone["color"],
            "fillOpacity": 0.30,
            "weight": 2
        }
    ))

# 2. Print exclusive band areas
radii = [0] + [z["radius_km"] for z in zones]

band_areas = []

print(f"{'Band':<20} {'Exclusive area':>18}")
print("-" * 40)

for i in range(1, len(radii)):
    r_inner = radii[i - 1]
    r_outer = radii[i]

    area_km2 = math.pi * (r_outer**2 - r_inner**2)
    band_areas.append((f"{r_inner}-{r_outer} km", area_km2))

    print(f"{r_inner}-{r_outer} km{'':<8} {area_km2:>14,.0f} km²")


# 3. Identify largest band
largest_band = max(band_areas, key=lambda x: x[1])

print()
print(f"Largest band: {largest_band[0]}")
print("It is largest because area grows with the square of the radius, so outer rings cover much more space.")

m

Band                     Exclusive area
----------------------------------------
0-75 km                 17,671 km²
75-200 km                107,992 km²
200-450 km                510,509 km²

Largest band: 200-450 km
It is largest because area grows with the square of the radius, so outer rings cover much more space.


Map(center=[21.305, -157.855], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…

## Exercise B

Create a 400 km buffer around **each** of the four targets (Tehran, Honolulu, Madrid, Riyadh) and display them all on a world map.

1. Use `LayersControl` so each target's buffer can be toggled.
2. Identify by inspection which pairs of buffers overlap.
3. Print the haversine distance between the two closest targets to confirm whether they should overlap.

In [7]:
import math
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

# Haversine distance function
def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlam / 2) ** 2
    )

    return R * 2 * math.asin(math.sqrt(a))


# Target coordinates
targets = {
    "Tehran":   [51.388, 35.695],
    "Honolulu": [-157.855, 21.305],
    "Madrid":   [-3.703, 40.417],
    "Riyadh":   [46.675, 24.688],
}

radius_km = 400

colors = {
    "Tehran": "#e74c3c",
    "Honolulu": "#2980b9",
    "Madrid": "#27ae60",
    "Riyadh": "#8e44ad",
}

# Create map
m = Map(center=(30, 20), zoom=2, basemap=basemaps.CartoDB.Positron)

# 1. Add a 400 km buffer around each target
for name, coords in targets.items():

    lon, lat = coords

    buffer_feat = buffer_feature(
        lon,
        lat,
        radius_km=radius_km,
        name=f"{name} 400 km"
    )

    point_feat = {
        "type": "Feature",
        "properties": {"name": name},
        "geometry": {
            "type": "Point",
            "coordinates": coords
        }
    }

    fc = {
        "type": "FeatureCollection",
        "features": [buffer_feat, point_feat]
    }

    m.add(GeoJSON(
        data=fc,
        name=f"{name} buffer",
        style={
            "color": colors[name],
            "fillColor": colors[name],
            "fillOpacity": 0.25,
            "weight": 2
        }
    ))

m.add(LayersControl())

# 2-3. Compare distances between all target pairs
print("Distances between target pairs:")
print("-" * 50)

closest_pair = None
closest_distance = float("inf")

target_names = list(targets.keys())

for i in range(len(target_names)):

    for j in range(i + 1, len(target_names)):

        name1 = target_names[i]
        name2 = target_names[j]

        lon1, lat1 = targets[name1]
        lon2, lat2 = targets[name2]

        distance = haversine_km(lon1, lat1, lon2, lat2)

        should_overlap = distance <= radius_km * 2

        print(
            f"{name1} → {name2}: "
            f"{distance:,.0f} km  "
            f"{'OVERLAP' if should_overlap else 'no overlap'}"
        )

        if distance < closest_distance:
            closest_distance = distance
            closest_pair = (name1, name2)

print()
print(f"Closest pair: {closest_pair[0]} and {closest_pair[1]}")
print(f"Distance: {closest_distance:,.0f} km")

if closest_distance <= radius_km * 2:
    print("These buffers should overlap.")
else:
    print("These buffers should not overlap.")

# Display map
m

Distances between target pairs:
--------------------------------------------------
Tehran → Honolulu: 12,969 km  no overlap
Tehran → Madrid: 4,774 km  no overlap
Tehran → Riyadh: 1,305 km  no overlap
Honolulu → Madrid: 12,649 km  no overlap
Honolulu → Riyadh: 14,255 km  no overlap
Madrid → Riyadh: 4,960 km  no overlap

Closest pair: Tehran and Riyadh
Distance: 1,305 km
These buffers should not overlap.


Map(center=[30, 20], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

---

## Check Your Understanding

**1.** What does overlap between two buffers from different targets represent in a real-world context?

**2.** Why do we compare multiple buffer sizes rather than picking one radius and using it for everything?

```python
# No code needed — answer in your own words
```

## Check Your Understanding

### 1.
What does overlap between two buffers from different targets represent in a real-world context?

An overlap means the influence zones of the two targets intersect.

In real-world applications, this could represent:
- shared risk areas
- overlapping warning zones
- overlapping communication coverage
- intersecting surveillance or radar ranges
- areas affected by multiple events or facilities

For example, if two 400 km emergency response zones overlap, people inside the overlap area are within range of both targets.

The overlap region often represents an area requiring special attention because it is affected by more than one source.

---

### 2.
Why do we compare multiple buffer sizes rather than picking one radius and using it for everything?

Different buffer sizes represent different levels of impact, uncertainty, or operational range.

One radius is usually too simplistic because real-world effects rarely stop at a single sharp boundary.

Examples:
- a small radius may represent severe impact
- a medium radius may represent moderate damage
- a larger radius may represent warning or monitoring range

Using multiple buffers helps show:
- intensity gradients
- uncertainty ranges
- different operational thresholds
- layered risk zones

It also allows analysts to compare how results change as the radius increases.

## Next

In [04 — Buffer Visualization Strategies](./04-Buffer_Visualization_Strategies.ipynb), we focus on how to style and order buffer layers so the map communicates clearly — avoiding the common mistakes that hide results or mislead interpretation.